# Document Processing and Multimodal RAG

**Title:** Document Processing and Multimodal RAG  
**Difficulty:** Expert  
**Notebook:** 09 of 09  

---

> *Handling real-world documents: PDFs, CSVs, JSON, HTML, and multimodal content in RAG systems.*

Real-world knowledge bases are not plain text files. They contain PDFs, spreadsheets, web pages, databases, images, and charts. This notebook teaches you how to process these diverse formats and build RAG systems that work with real data.

## Learning Objectives

After this notebook you will be able to:

1. **Load** documents from multiple formats (TXT, PDF, CSV, JSON, HTML, Markdown)
2. **Understand** the Document object and its metadata
3. **Build** PDF RAG pipelines with source attribution
4. **Handle** structured data (CSV, JSON) appropriately
5. **Understand** multimodal RAG concepts and architectures
6. **Build** a Data Science Course Knowledge Assistant from mixed sources
7. **Apply** security practices for untrusted documents

## Prerequisites

| Concept | Source |
|---------|--------|
| Basic RAG pipeline | Notebook 05 |
| Advanced RAG techniques | Notebook 08 |
| Embeddings and vector stores | Notebook 04 |

> This notebook builds on Notebooks 05 and 08.

## Setup

In [ ]:
import os
import json
import csv
import numpy as np
import pandas as pd
from pathlib import Path
from io import StringIO
from dotenv import load_dotenv

from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma

load_dotenv()
print('All imports loaded!')

In [ ]:
api_key = os.getenv('OPENAI_API_KEY', '')
ollama_available = False
try:
    import requests
    r = requests.get('http://localhost:11434/api/tags', timeout=2)
    ollama_available = r.status_code == 200
except: pass

print(f'OpenAI API: {"available" if api_key else "NOT SET"}')
print(f'Ollama: {"available" if ollama_available else "NOT RUNNING"}')

---

## 1. Why Document Loading Matters

Real-world knowledge bases contain diverse file formats:

| Format | Use Case | Challenge |
|--------|----------|-----------|
| **TXT** | Plain text notes | Simple, but no structure |
| **Markdown** | Documentation, notes | Headers, code blocks |
| **PDF** | Academic papers, reports | Complex layout, tables, images |
| **CSV** | Datasets, structured data | Tabular, not text paragraphs |
| **JSON** | APIs, config files | Nested structure |
| **HTML** | Web pages, documentation | Navigation, ads, boilerplate |
| **DOCX** | Word documents, reports | Formatting, tables, images |

### The Document Loading Pipeline

```mermaid
graph TD
    F[File] --> L[Loader]
    L --> D[Document]
    D --> M[Metadata]
    D --> TS[Text Splitting]
    TS --> E[Embedding]
    E --> VS[Vector Store]
```

### Key Principle

> *The right loader for the right format. Do not force everything through a text parser.*

---

## 2. The Document Object

Every loaded file becomes a `Document` with two key attributes:

| Attribute | Type | Description |
|-----------|------|-------------|
| `page_content` | `str` | The text content |
| `metadata` | `dict` | Source info, page numbers, custom fields |

In [ ]:
# Create a Document manually
doc = Document(
    page_content='Random Forest is an ensemble method that builds multiple decision trees.',
    metadata={'source': 'ml_notes.md', 'topic': 'ensemble methods', 'difficulty': 'intermediate'}
)

print(f'Content: {doc.page_content}')
print(f'Metadata: {doc.metadata}')
print(f'Source: {doc.metadata["source"]}')

### Why Metadata Matters in RAG

- **Source attribution**: Tell users which document the answer came from
- **Filtering**: Search only within specific topics or difficulty levels
- **Debugging**: Identify which documents are retrieved
- **Security**: Track provenance of information

---

## 3. Loading Text and Markdown Files

The simplest loaders read files directly into Documents.

In [ ]:
# Create sample files
data_dir = Path('../data/sample_docs')
data_dir.mkdir(parents=True, exist_ok=True)

# Text file
(data_dir / 'intro_ml.txt').write_text(
    'Machine Learning is a subset of Artificial Intelligence that enables systems to learn from data.\n'
    'ML algorithms build models from training data to make predictions or decisions.\n'
    'The three main types are supervised learning, unsupervised learning, and reinforcement learning.')

# Markdown file
(data_dir / 'notes.md').write_text(
    '# Data Science Notes\n\n'
    '## Classification\n'
    'Classification predicts discrete labels. Common algorithms include Logistic Regression, Random Forest, and SVM.\n\n'
    '## Regression\n'
    'Regression predicts continuous values. Use Linear Regression, Ridge, or gradient boosting methods.')

# HTML-like content
(data_dir / 'web_notes.html').write_text(
    '<html><body><h1>Pandas Tutorial</h1>\n'
    '<p>Pandas provides DataFrame for tabular data manipulation.</p>\n'
    '<p>Key functions: read_csv(), describe(), groupby(), merge().</p>\n'
    '</body></html>')

print('Sample files created!')

In [ ]:
# Method 1: Manual loading (works everywhere, no extra packages)
def load_text_file(filepath):
    text = Path(filepath).read_text(encoding='utf-8')
    return Document(page_content=text, metadata={'source': str(filepath), 'type': 'text'})

def load_markdown_file(filepath):
    text = Path(filepath).read_text(encoding='utf-8')
    return Document(page_content=text, metadata={'source': str(filepath), 'type': 'markdown'})

def load_html_file(filepath):
    import re
    text = Path(filepath).read_text(encoding='utf-8')
    # Simple HTML tag removal
    text = re.sub('<[^<]+?>', ' ', text)
    text = re.sub('\s+', ' ', text).strip()
    return Document(page_content=text, metadata={'source': str(filepath), 'type': 'html'})

docs = [
    load_text_file(data_dir / 'intro_ml.txt'),
    load_markdown_file(data_dir / 'notes.md'),
    load_html_file(data_dir / 'web_notes.html'),
]

for d in docs:
    print(f'[{d.metadata["type"]}] {d.metadata["source"]}')
    print(f'  Content: {d.page_content[:80]}...')
    print()

---

## 4. PDF Processing

PDFs are the most common format for academic and business documents.

### PDF RAG Pipeline

```mermaid
graph TD
    PDF[PDF File] --> PL[PyPDFLoader]
    PL --> PG[Page Documents]
    PG --> SP[Split into Chunks]
    SP --> EM[Embeddings]
    EM --> VS[Vector Store]
    VS --> R[Retriever]
    R --> LLM[LLM]
```

### Creating a Sample PDF

We will create a small educational PDF for demonstration.

In [ ]:
# Create a sample PDF using fpdf2
# If fpdf2 is not installed, we simulate the content

try:
    from fpdf import FPDF

    pdf = FPDF()
    pdf.add_page()
    pdf.set_font('Arial', 'B', 16)
    pdf.cell(200, 10, txt='Introduction to Machine Learning', ln=True, align='C')
    pdf.set_font('Arial', '', 12)

    sections = [
        ('Chapter 1: What is ML?', 'Machine Learning is a branch of AI that creates systems that learn from data. Instead of being explicitly programmed, these systems improve through experience. ML is used in recommendation systems, fraud detection, and autonomous vehicles.'),
        ('Chapter 2: Supervised Learning', 'In supervised learning, the model learns from labeled training data. The goal is to learn a mapping from inputs to outputs. Classification predicts categories (spam/not spam). Regression predicts continuous values (house prices).'),
        ('Chapter 3: Model Evaluation', 'Never evaluate on training data. Use a train/test split. For classification: accuracy, precision, recall, F1 score. For regression: MSE, RMSE, MAE, R-squared. Cross-validation gives robust estimates.'),
        ('Chapter 4: Feature Engineering', 'Feature engineering creates new input variables from raw data. Common techniques: one-hot encoding for categories, scaling for numerical features, log transforms for skewed data, and interaction features.'),
    ]

    for title, content in sections:
        pdf.set_font('Arial', 'B', 14)
        pdf.cell(200, 10, txt=title, ln=True)
        pdf.set_font('Arial', '', 11)
        pdf.multi_cell(0, 7, txt=content)
        pdf.ln(5)

    pdf_path = str(data_dir / 'ml_textbook.pdf')
    pdf.output(pdf_path)
    print(f'PDF created: {pdf_path}')

except ImportError:
    print('fpdf2 not installed. Using simulated PDF content.')
    pdf_path = None

In [ ]:
# Load PDF using PyPDFLoader (from langchain_community)
# Note: langchain_community is archived but loaders still work and are widely used

try:
    from langchain_community.document_loaders import PyPDFLoader

    if pdf_path:
        loader = PyPDFLoader(pdf_path)
        pdf_docs = loader.load()
        print(f'Loaded {len(pdf_docs)} pages from PDF')
        for d in pdf_docs:
            print(f'  Page {d.metadata.get("page", "?")}: {d.page_content[:80]}...')
    else:
        print('PDF not created. Skipping PyPDFLoader demo.')

except ImportError:
    print('PyPDFLoader requires langchain-community. Install: pip install langchain-community pypdf')
    # Fallback: create documents from the content we know is in the PDF
    pdf_docs = [Document(
        page_content='Machine Learning is a branch of AI that creates systems that learn from data.',
        metadata={'source': 'ml_textbook.pdf', 'page': 1})
    ]
    print('Using simulated PDF content.')

In [ ]:
# Build PDF RAG
if pdf_docs:
    splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
    chunks = splitter.split_documents(pdf_docs)

    embeddings = OpenAIEmbeddings(model='text-embedding-3-small')
    vs = Chroma.from_documents(chunks, embeddings)

    # Test retrieval
    query = 'What is supervised learning?'
    results = vs.similarity_search(query, k=2)

    print(f'Query: {query}')
    print(f'Retrieved {len(results)} documents:')
    for i, d in enumerate(results):
        print(f'  {i+1}. [Page {d.metadata.get("page","?")}] {d.page_content[:100]}...')

---

## 5. CSV Data: Document vs Analysis

CSV files require a decision: **retrieval** or **analysis**?

| Approach | When to Use | Example |
|----------|-------------|---------|
| **Document retrieval** | Finding relevant rows by meaning | "Which patients have condition X?" |
| **Structured analysis** | Numerical computations | "What is the average salary?" |

```mermaid
graph TD
    CSV[CSV File] --> Decision{Question Type?}
    Decision -->|Semantic/Language| DR[Document Retrieval]
    Decision -->|Numerical/Statistical| SA[Structured Analysis]
    DR --> LLM[LLM Answer]
    SA --> Pandas[Pandas Answer]
```

> **Important**: Do NOT use an LLM to calculate the mean of 1000 numbers. Use Pandas.

In [ ]:
# Create sample dataset
employees = pd.DataFrame({
    'name': ['Alice', 'Bob', 'Carol', 'David', 'Eve', 'Frank', 'Grace', 'Hank'],
    'department': ['Engineering', 'Marketing', 'Engineering', 'Sales', 'Marketing', 'Engineering', 'Sales', 'Marketing'],
    'salary': [95000, 72000, 105000, 68000, 78000, 110000, 65000, 82000],
    'experience_years': [5, 3, 8, 2, 4, 10, 1, 6],
    'performance': ['excellent', 'good', 'excellent', 'average', 'good', 'excellent', 'average', 'good']
})

csv_path = data_dir / 'employees.csv'
employees.to_csv(csv_path, index=False)
print(f'CSV created: {csv_path}')
print(employees)

In [ ]:
# Document retrieval approach: each row becomes a document
def csv_to_documents(filepath):
    docs = []
    with open(filepath, 'r') as f:
        reader = csv.DictReader(f)
        for i, row in enumerate(reader):
            # Create a natural language description of each row
            content = f"Employee {row['name']} works in {row['department']} with {row['experience_years']} years experience and salary of ${row['salary']}. Performance: {row['performance']}."
            docs.append(Document(page_content=content,
                metadata={'source': str(filepath), 'row': i, 'name': row['name']}))
    return docs

csv_docs = csv_to_documents(csv_path)

print(f'Loaded {len(csv_docs)} documents from CSV')
for d in csv_docs[:3]:
    print(f'  {d.page_content}')

In [ ]:
# Structured analysis approach: use Pandas for numerical queries
print('=== When to use Pandas (NOT LLM) ===')

print(f'\nAverage salary: ${employees["salary"].mean():,.0f}')
print(f'Highest salary: ${employees["salary"].max():,.0f} ({employees.loc[employees["salary"].idxmax(), "name"]})')
print(f'Department counts: {employees["department"].value_counts().to_dict()}')
print(f'Avg salary by dept:\n{employees.groupby("department")["salary"].mean().to_string()}')

print('\nRule of thumb:')
print('  - Numbers, statistics, aggregation -> Use Pandas')
print('  - Find similar rows, semantic search -> Use Document retrieval')
print('  - Both -> Build a router that detects the query type')

---

## 6. JSON and Structured Content

JSON files have nested structure. The key decision is how to flatten them into Documents.

In [ ]:
# Create sample JSON knowledge base
json_data = {
    'algorithms': [
        {'name': 'Random Forest', 'type': 'ensemble', 'use_case': 'Classification and regression', 'strengths': 'Robust, handles missing data', 'weaknesses': 'Less interpretable'},
        {'name': 'XGBoost', 'type': 'boosting', 'use_case': 'Tabular data competitions', 'strengths': 'High accuracy, regularization', 'weaknesses': 'Requires tuning'},
        {'name': 'K-Means', 'type': 'clustering', 'use_case': 'Customer segmentation', 'strengths': 'Simple, fast', 'weaknesses': 'Must specify k'},
    ]
}

(data_dir / 'algorithms.json').write_text(json.dumps(json_data, indent=2))
print('JSON file created.')

In [ ]:
# Load JSON by flattening each entry into a Document
def json_to_documents(filepath):
    data = json.loads(Path(filepath).read_text())
    docs = []
    for category, items in data.items():
        for item in items:
            # Flatten to text
            content = f"{item['name']}: {item['type']} algorithm. Use case: {item['use_case']}. Strengths: {item['strengths']}. Weaknesses: {item['weaknesses']}."
            docs.append(Document(
                page_content=content,
                metadata={'source': str(filepath), 'category': category, 'name': item['name'], 'type': item['type']}))
    return docs

json_docs = json_to_documents(data_dir / 'algorithms.json')

for d in json_docs:
    print(f'[{d.metadata["type"]}] {d.page_content}')

---

## 7. Multimodal RAG

Real-world documents contain more than text: images, tables, charts, and diagrams.

### What is Multimodal RAG?

```mermaid
graph TD
    T[Text] --> KB[Knowledge Base]
    I[Images] --> KB
    TB[Tables] --> KB
    CH[Charts] --> KB
    KB --> R[Retriever]
    R --> MM[Vision-Language Model]
    MM --> A[Answer]
```

### Approaches to Multimodal RAG

| Approach | Description | Complexity |
|----------|-------------|------------|
| **Text-only** | Extract text, ignore images | Low |
| **Image descriptions** | Generate text descriptions of images, then RAG on text | Medium |
| **Vision models** | Use GPT-4o or similar to understand images directly | Medium |
| **Table extraction** | Parse tables to structured data, query with SQL/Pandas | Medium |
| **Full multimodal** | Embed images and text together | High |

### Vision-Language Models

Models like GPT-4o and Llama 3.2 Vision can understand images:

- **Chart understanding**: Read and interpret charts/graphs
- **Table extraction**: Convert image tables to structured data
- **Document layout**: Understand page structure
- **Code screenshots**: Read code from images

In [ ]:
# Conceptual example: Vision model for chart understanding
# This requires a vision-capable model (GPT-4o, Claude 3.5, etc.)

# For demonstration, we show the architecture:

multimodal_rag_architecture = """

MULTIMODAL RAG ARCHITECTURE

1. Document Ingestion:
   - Text files -> Text chunks
   - PDFs -> Page text + images
   - CSVs -> Row descriptions + raw data
   - Images -> Vision model descriptions

2. Storage:
   - Text embeddings -> Vector store
   - Image descriptions -> Vector store
   - Original images -> File system or object storage

3. Retrieval:
   - Query -> Text search + Image description search
   - Combine results from both modalities

4. Generation:
   - Text context -> LLM prompt
   - Image context -> Vision model prompt
   - Combined answer -> Final response
"""

print(multimodal_rag_architecture)

In [ ]:
# Practical example: Table-aware RAG
# When a document contains tables, we can extract and query them separately

# Create a sample table document
table_content = """
Performance Comparison of ML Algorithms

| Algorithm | Accuracy | Training Time | Interpretability |
|-----------|----------|---------------|------------------|
| Logistic Regression | 82% | Fast | High |
| Random Forest | 88% | Medium | Medium |
| XGBoost | 91% | Medium | Low |
| Neural Network | 89% | Slow | Very Low |

Key findings:
- XGBoost achieves highest accuracy on this dataset
- Logistic Regression is most interpretable
- Random Forest offers good balance
"""

table_doc = Document(page_content=table_content,
    metadata={'source': 'benchmarks.md', 'type': 'table_report', 'contains_table': True})

print('Table document created.')
print(f'Content length: {len(table_doc.page_content)} chars')

---

## 8. Complete Example: Data Science Course Knowledge Assistant

Let us build a knowledge assistant that handles multiple document types:

```mermaid
graph TD
    TXT[Text Notes] --> Loader[Multi-Format Loader]
    MD[Markdown Notes] --> Loader
    PDF[PDF Textbook] --> Loader
    CSV[CSV Datasets] --> Loader
    JSON[JSON Reference] --> Loader
    Loader --> TS[Text Splitter]
    TS --> VS[Vector Store]
    VS --> R[Retriever]
    R --> LLM[LLM]
    LLM --> A[Answer with Sources]
```

In [ ]:
class MultiFormatKnowledgeBase:
    """Load and query documents from multiple formats."""

    def __init__(self, embeddings_model):
        self.embeddings = embeddings_model
        self.all_docs = []
        self.vs = None

    def load_text(self, filepath):
        text = Path(filepath).read_text(encoding='utf-8')
        self.all_docs.append(Document(page_content=text,
            metadata={'source': str(filepath), 'format': 'text'}))

    def load_markdown(self, filepath):
        text = Path(filepath).read_text(encoding='utf-8')
        self.all_docs.append(Document(page_content=text,
            metadata={'source': str(filepath), 'format': 'markdown'}))

    def load_csv_as_docs(self, filepath):
        with open(filepath, 'r') as f:
            reader = csv.DictReader(f)
            for i, row in enumerate(reader):
                content = ', '.join(f'{k}: {v}' for k, v in row.items())
                self.all_docs.append(Document(page_content=content,
                    metadata={'source': str(filepath), 'format': 'csv', 'row': i}))

    def load_json(self, filepath):
        data = json.loads(Path(filepath).read_text())
        for category, items in data.items():
            if isinstance(items, list):
                for item in items:
                    content = ', '.join(f'{k}: {v}' for k, v in item.items())
                    self.all_docs.append(Document(page_content=content,
                        metadata={'source': str(filepath), 'format': 'json', 'category': category}))

    def build(self, chunk_size=500):
        splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=50)
        chunks = splitter.split_documents(self.all_docs)
        self.vs = Chroma.from_documents(chunks, self.embeddings)
        print(f'Built knowledge base: {len(self.all_docs)} docs -> {len(chunks)} chunks')

    def query(self, question, k=3):
        if not self.vs: return 'Knowledge base not built yet.'
        docs = self.vs.similarity_search(question, k=k)

        prompt = ChatPromptTemplate.from_messages([
            ('system', 'Answer using ONLY the provided context. Cite the source document.'),
            ('human', 'Context:\n{context}\n\nQuestion: {question}\n\nAnswer:')])
        chain = ({'context': lambda x: '\n\n'.join(d.page_content for d in x['docs']),
                  'question': RunnablePassthrough()}
                 | prompt | ChatOpenAI(model='gpt-4o-mini', temperature=0) | StrOutputParser())

        answer = chain.invoke({'docs': docs, 'question': question})
        return {'answer': answer, 'sources': list(set(d.metadata['source'] for d in docs))}

# Build the knowledge base
kb = MultiFormatKnowledgeBase(OpenAIEmbeddings(model='text-embedding-3-small'))

# Load all formats
kb.load_text(data_dir / 'intro_ml.txt')
kb.load_markdown(data_dir / 'notes.md')
kb.load_csv_as_docs(csv_path)
kb.load_json(data_dir / 'algorithms.json')

kb.build()

In [ ]:
# Test the knowledge assistant
questions = [
    'What is the best algorithm for tabular data?',
    'What is supervised learning?',
    'What is the average salary in the Engineering department?',
]

for q in questions:
    print(f'Q: {q}')
    result = kb.query(q)
    print(f'A: {result["answer"][:200]}...')
    print(f'Sources: {result["sources"]}')
    print()

---

## 9. Local Ollama Implementation

The same system works with Ollama. For PDF processing, the loaders are format-based and do not depend on the LLM provider.

```bash
# Prerequisites:
ollama --version
ollama pull llama3.2
ollama pull nomic-embed-text
ollama list
```

In [ ]:
# Ollama version - only change the LLM and embeddings
if ollama_available:
    local_embeddings = OllamaEmbeddings(model='nomic-embed-text')
    local_llm = ChatOllama(model='llama3.2', temperature=0)

    local_kb = MultiFormatKnowledgeBase(local_embeddings)
    local_kb.load_text(data_dir / 'intro_ml.txt')
    local_kb.load_markdown(data_dir / 'notes.md')
    local_kb.load_csv_as_docs(csv_path)
    local_kb.load_json(data_dir / 'algorithms.json')
    local_kb.build()

    # Override query method to use local LLM
    def local_query(question, k=3):
        docs = local_kb.vs.similarity_search(question, k=k)
        prompt = ChatPromptTemplate.from_messages([
            ('system', 'Answer using ONLY the provided context.'),
            ('human', 'Context:\n{context}\n\nQ: {question}\n\nA:')])
        chain = ({'context': lambda x: '\n\n'.join(d.page_content for d in x['docs']),
                  'question': RunnablePassthrough()}
                 | prompt | local_llm | StrOutputParser())
        return chain.invoke({'docs': docs, 'question': question})

    print(local_query('What is Random Forest?')[:200])
else:
    print('Ollama not available. Run ollama serve first')

---

## 10. Security: Untrusted Documents

When loading documents from external sources, treat them as **untrusted**.

### Threats

| Threat | Description | Example |
|--------|-------------|---------|
| **Prompt injection** | Malicious text in documents tricks the LLM | Document contains "Ignore previous instructions..." |
| **Metadata poisoning** | Fake metadata misleads retrieval | Setting source to a trusted domain falsely |
| **Malicious PDFs** | PDFs with embedded scripts or exploits | crafted to crash loaders or execute code |
| **Sensitive data** | Documents contain PII or secrets | Leaked API keys, personal information |
| **Resource exhaustion** | Extremely large files crash the system | Gigabyte-sized PDFs or CSVs |

### Defensive Measures

1. **Input validation**: Check file size, format, encoding before processing
2. **Sandboxing**: Run loaders in isolated environments
3. **Content filtering**: Remove or flag suspicious patterns
4. **Size limits**: Set maximum file and chunk sizes
5. **Source verification**: Only load from trusted directories
6. **Monitoring**: Log all loaded documents for audit

In [ ]:
# Security best practices for document loading
def safe_load_file(filepath, max_size_mb=10):
    """Load a file with security checks."""
    fp = Path(filepath)

    # Check file exists
    if not fp.exists():
        raise FileNotFoundError(f'File not found: {filepath}')

    # Check file size
    size_mb = fp.stat().st_size / (1024 * 1024)
    if size_mb > max_size_mb:
        raise ValueError(f'File too large: {size_mb:.1f}MB (max {max_size_mb}MB)')

    # Check file extension
    allowed = {'.txt', '.md', '.csv', '.json', '.html'}
    if fp.suffix.lower() not in allowed:
        raise ValueError(f'Unsupported file type: {fp.suffix}')

    # Read with encoding
    text = fp.read_text(encoding='utf-8', errors='replace')

    # Basic content sanitization
    suspicious = ['ignore previous instructions', 'system prompt', 'you are now']
    for pattern in suspicious:
        if pattern.lower() in text.lower():
            print(f'WARNING: Suspicious pattern detected in {filepath}')

    return Document(page_content=text, metadata={'source': str(filepath), 'safe_loaded': True})

# Test
doc = safe_load_file(data_dir / 'intro_ml.txt')
print(f'Loaded safely: {doc.metadata["source"]}')

---

## 11. Exercises

### Exercise 1: Add DOCX Support

Create a function `load_docx(filepath)` that extracts text from Word documents. Hint: use the `python-docx` library.

### Exercise 2: Multi-Format Router

Build a query router that detects whether a question is:
- **Numerical** -> Use Pandas for calculation
- **Semantic** -> Use vector store for retrieval

Test with questions like:
- "What is the average salary?" (numerical)
- "Who is the most experienced engineer?" (semantic)

### Exercise 3: Source Attribution

Modify the knowledge assistant to include the source document name and page number in every answer.

### Challenge 1: Incremental Loading

Build a system that watches a directory for new files and automatically adds them to the knowledge base.

### Challenge 2: Document Quality Scoring

Implement a quality score for each document based on:
- Length (too short = low quality)
- Readability
- Relevance to the domain

Use the quality score during retrieval to prefer high-quality documents.

### Mini-Project: Course Materials RAG

Create a complete RAG system for a university course that handles:
- Lecture notes (Markdown)
- Textbook chapters (PDF)
- Dataset descriptions (CSV)
- Algorithm references (JSON)

Include source attribution and handle out-of-domain questions gracefully.

---

## 12. Key Takeaways

| Format | Loader Approach | Key Consideration |
|--------|----------------|-------------------|
| **TXT/MD** | Direct file read | Simplest format |
| **PDF** | PyPDFLoader / Unstructured | Layout, tables, images |
| **CSV** | Pandas or row-to-doc | Decide: retrieval vs analysis |
| **JSON** | Flatten to text | Preserve structure in metadata |
| **HTML** | Strip tags, extract content | Remove boilerplate |

### Core Principles

1. **Right loader for the right format** -- do not force everything through text parsing
2. **Metadata is critical** -- source attribution, filtering, debugging
3. **CSV needs a router** -- numerical questions use Pandas, semantic questions use retrieval
4. **Multimodal is the future** -- text-only RAG misses images, tables, charts
5. **Security first** -- validate, sanitize, limit size, monitor loaded documents

### The Complete Document Pipeline

```
File -> Loader -> Document -> Metadata -> Split -> Embed -> Store -> Retrieve -> Generate
```

### Complete Repository Progress

```
01. Introduction         -- What is LangChain?
02. Models & Prompts     -- Chat models, messages, templates
03. LCEL & Chains        -- Pipeline composition
04. Embeddings           -- Vector representations
05. Basic RAG            -- Retrieval-augmented generation
06. Tools & Agents       -- Dynamic tool calling
07. Capstone             -- Data Science AI Tutor
08. Advanced RAG         -- Chunking, reranking, evaluation
09. Document Processing  -- PDFs, CSVs, JSON, multimodal
```

> *You now have a complete LangChain toolkit for building production-grade RAG systems with real-world documents.*